# 环节 02 · 最小 Transformer 与信任底座（配套 Notebook）

> 配套长文：[环节02-最小Transformer与信任底座详解.md](./环节02-最小Transformer与信任底座详解.md) · 导航：[环节00](./环节00-总揽与环节导航.md)
> 定位：四个词排成正方形；对比自由 8 参 / 对角 4 参 / 矩形 2 参。**纯标准库。**

| 本 Notebook | 长文章节 | 验证什么 |
|---|---|---|
| §1 正方形嵌入 | §2 | 人/动词 × 友善/不友善 |
| §2 对角约束 | §3 / §5 | 8→4；增量是外积 |
| §3 矩形约束 | §3 | 只调宽高 = 2 个旋钮 |
| §4 旋钮对照 | §3 | 你信任的结构 = 你不让动的轴 |


## 1. 底座已经排好的正方形（长文 §2）

Mary / Bob / ignored / greeted。横轴：右=人、左=动词；纵轴：上=友善、下=不友善。


In [ ]:
WORDS = ("Mary", "Bob", "ignored", "greeted")
W = {
    "Mary":    (1.0,  1.0),   # 人, 友善
    "Bob":     (1.0, -1.0),   # 人, 不友善
    "ignored": (-1.0, -1.0),  # 动词, 不友善
    "greeted": (-1.0,  1.0),  # 动词, 友善
}

def show(title, pts):
    print(title)
    for w in WORDS:
        x, y = pts[w]
        print(f"  {w:8} ({x:+5.1f}, {y:+5.1f})")
    print()

show("底座 W（正方形）", W)
print("自由微调：每点横+纵，共 4×2 = 8 个旋钮。")


## 2. 对角约束：沿同一斜率移动（长文 §3 / §5）

斜率 (p, q)=(1, 1) 即 45°。四点各走 a,b,c,d。增量矩阵 = 列向量 × 行向量，秩 1。


In [ ]:
def add(pts, delta):
    return {w: (pts[w][0] + delta[w][0], pts[w][1] + delta[w][1]) for w in WORDS}

def diagonal_delta(a, b, c, d, p=1.0, q=1.0):
    """Δ = [a,b,c,d]ᵀ · [p, q]"""
    dist = {"Mary": a, "Bob": b, "ignored": c, "greeted": d}
    return {w: (dist[w] * p, dist[w] * q) for w in WORDS}

delta = diagonal_delta(0.5, -0.2, 0.1, 0.3)
W2 = add(W, delta)
show("对角微调后（斜率 1，4 个旋钮）", W2)

print("增量矩阵 Δ（应每行成比例）：")
for w in WORDS:
    dx, dy = delta[w]
    print(f"  {w:8} [{dx:+4.1f}  {dy:+4.1f}]")

rows = [delta[w] for w in WORDS]
ok = all(abs(dx * 1.0 - dy * 1.0) < 1e-9 for dx, dy in rows)
print(f"\n每行都是 [1, 1] 的倍数？ {ok}  → 秩 1（环节 03）")
print("旋钮：a,b,c,d 四个，不是八个。")


## 3. 矩形约束：信任两条语义轴，只调宽和高（长文 §3）

「不保证还是正方形，但信任矩形性。」人仍在右、动词仍在左、友善仍在上——只拉长宽。2 个旋钮。


In [ ]:
def rectangle(half_w, half_h):
    return {
        "Mary":    ( half_w,  half_h),
        "Bob":     ( half_w, -half_h),
        "ignored": (-half_w, -half_h),
        "greeted": (-half_w,  half_h),
    }

show("瘦高矩形（宽 0.5，高 2）", rectangle(0.5, 2.0))
show("扁宽矩形（宽 2，高 0.5）", rectangle(2.0, 0.5))
print("轴的含义没变，只是刻度变了。这就是「信任底座已经发现的结构」。")
print("自由微调 8 参可以打破矩形（把人拧到动词那边）——LoRA 故意不让你这么做。")


## 4. 旋钮对照（长文 §3）


In [ ]:
rows = [
    ("什么都不信，四点任意", 8, "整个平面"),
    ("更新沿同一斜率（对角 / 外积）", 4, "四条平行轨"),
    ("斜率也学（外积 + 尺度）", 5, "方向 1 + 四点位置；见环节 04"),
    ("信任矩形性，只调宽高", 2, "各种矩形"),
    ("信任圆，只调四个角", 4, "沿圆周滑"),
]
print(f"{'你信任什么':<28} {'旋钮':>6}  点怎么动")
print("-" * 60)
for name, n, how in rows:
    print(f"{name:<28} {n:>6}  {how}")

print("\nLoRA 合法 ⟺ 起点已经很好。随机初始化上不要做 LoRA。")
